# Inference / Testing Notebook (Straightforward)

This notebook is **inference-only** (no training).  
It assumes your repository already contains the `model/`, `utilities/` and `classifier_rev.py/` .

**What it does**
1. Adds the repo root to `sys.path`
2. Loads a Lightning checkpoint (`.ckpt`)
3. Rebuilds the `Classifier` using checkpoint hyperparameters (with optional path overrides)
4. Runs `trainer.test(...)` to execute your `test_step` / `on_test_epoch_end`
5. Reads the saved patient-level outputs (`patient_level_fold*.npz`) and shows a quick summary

> If your `Classifier` lives in a different module than shown below, change the import in Cell 2.


In [1]:

import os
import sys
from pathlib import Path

# ====== REPO ROOT ======
# Put this notebook anywhere in the repo. This finds the repo root by walking up until it sees "model/".
HERE = Path.cwd()

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(10):
        if (p / "model").exists() and (p / "utilities").exists():
            return p
        p = p.parent
    return start.resolve()

REPO_ROOT = find_repo_root(HERE)
print("REPO_ROOT =", REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("sys.path[0] =", sys.path[0])


REPO_ROOT = /home/hadya/GPModels
sys.path[0] = /home/hadya/GPModels


In [2]:

# ====== Imports ======
import numpy as np
import torch
import pytorch_lightning as pl

# IMPORTANT:
# Update this import to wherever Classifier is defined in your repo.
from classifier_rev import Classifier  # <-- CHANGE ME if needed

print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)
print("CUDA available:", torch.cuda.is_available())


/home/hadya/miniconda3/envs/gpmodels311/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/hadya/miniconda3/envs/gpmodels311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 1701


Torch: 2.6.0
Lightning: 2.6.0
CUDA available: True


In [3]:

# ====== Config (edit these paths) ======

# 1) checkpoint to test/infer
CKPT_PATH = Path("/project/Data/GPModels/outputs/checkpoints/fold0/GPUNet/epoch=291-step=65407.ckpt")

# 2) override paths inside checkpoint if you run on a different machine
MAIN_PATH_OVERRIDE = "/project/Data/GPModels"
OUT_PATH_OVERRIDE  = "/project/Data/GPModels/outputs/notebookTest"

# 3) choose device
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES = 1 if torch.cuda.is_available() else "auto"

print("CKPT_PATH:", CKPT_PATH)
print("ACCELERATOR:", ACCELERATOR, "| DEVICES:", DEVICES)


CKPT_PATH: /project/Data/GPModels/outputs/checkpoints/fold0/GPUNet/epoch=291-step=65407.ckpt
ACCELERATOR: gpu | DEVICES: 1


In [4]:

# ====== Load model from checkpoint hyperparameters + state_dict ======
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

ckpt_hparams = ckpt.get("hyper_parameters") or ckpt.get("hparams")
if ckpt_hparams is None:
    raise KeyError("Checkpoint missing 'hyper_parameters'/'hparams'")

if not isinstance(ckpt_hparams, dict):
    ckpt_hparams = vars(ckpt_hparams)

# Optional path overrides
ckpt_hparams["main_path"] = MAIN_PATH_OVERRIDE
ckpt_hparams["out_path"]  = OUT_PATH_OVERRIDE

model = Classifier(**ckpt_hparams)

# Load weights
state_dict = ckpt["state_dict"]
model.load_state_dict(state_dict, strict=True)

# Put in eval mode (Lightning will also handle it during test)
model.eval()

print("Loaded model:", type(model).__name__)
print("trainID:", getattr(model.hparams, "trainID", None))
print("network:", getattr(model.hparams, "network", None))


Loaded model: Classifier
trainID: GPUNet_150e_None_fixDice_Brats20_full_allCont_Axi_GP_UNet_1-value_w_d-0.0005_drop-True_sinc_norm3_fold0of5
network: GP_UNet


In [ ]:

# ====== Run TEST (uses your test_step + on_test_epoch_end) ======

trainer = pl.Trainer(
    logger=False,
    enable_checkpointing=False,
    accelerator=ACCELERATOR,
    devices=DEVICES,
    precision=32,          # keep simple and deterministic
    deterministic=True,
)


# Make sure the dataset arrays exist
model.setup(stage="test")

dl = model.test_dataloader()
batch = next(iter(dl))

image, mask, label, gidx = batch
print(image.shape, mask.shape, label.shape, gidx.shape)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Loaded pickles:
X shape: (39163, 4, 240, 240)
m shape: (39163, 1, 240, 240)
y shape: (39163,)
Classes present: [0. 1. 2.]


In [ ]:
results = trainer.test(model=model, ckpt_path=None, verbose=True)
print("trainer.test results:", results)

# Your code saves outputs under:
#   OUT_PATH_OVERRIDE/Results/<trainID>/
out_dir = Path(OUT_PATH_OVERRIDE) / "Results" / str(model.hparams.trainID)
print("Expected results dir:", out_dir)

if out_dir.exists():
    # Show what was produced
    files = sorted([p.name for p in out_dir.glob("*")])
    print("Files in results dir:")
    for f in files:
        print(" -", f)
else:
    print("Results dir not found. Did your code write to out_path correctly?")


In [ ]:

# ====== Quick summary from patient-level npz (if present) ======
# Your on_test_epoch_end saves: patient_level_fold{fold_id}.npz in OUT_PATH/Results/trainID/  (or a sibling folder in your code)
# In your current code, it actually saves into: OUT_PATH/Results/trainID/  AND also out_dir built from OUT_PATH/Results/trainID at the end.
# We'll search recursively under out_dir.

import numpy as np

if not out_dir.exists():
    raise FileNotFoundError(out_dir)

npz_files = sorted(out_dir.rglob("patient_level_fold*.npz"))
print("Found NPZ files:", [p.name for p in npz_files])

if len(npz_files) == 0:
    print("No patient_level_fold*.npz found. Check your on_test_epoch_end saving paths.")
else:
    npz_path = npz_files[0]
    data = np.load(npz_path, allow_pickle=True)
    keys = list(data.keys())
    print("Using:", npz_path)
    print("Keys:", keys)

    true_grade = data["true_grade"]
    pred_grade = data["pred_grade"]
    pat_acc = (true_grade == pred_grade).mean() if len(true_grade) else float("nan")
    print(f"Patient-level LGG vs HGG accuracy: {pat_acc:.4f} (n={len(true_grade)})")

    # Optional metrics if present
    for k in ["pat_auc_HGG", "pat_ap_HGG", "pat_auc_LGG", "pat_ap_LGG"]:
        if k in keys:
            print(k, "=", data[k])

    # Dice arrays if present
    if "dice_raw_patient" in keys:
        dr = data["dice_raw_patient"]
        print("dice_raw_patient: mean=", np.nanmean(dr), "median=", np.nanmedian(dr))
    if "dice_clean_patient" in keys:
        dc = data["dice_clean_patient"]
        print("dice_clean_patient: mean=", np.nanmean(dc), "median=", np.nanmedian(dc))


In [ ]:

# ====== (Optional) Visualize a few slice-level predictions quickly ======
# This is a lightweight manual loop over the test dataloader.
# It DOES NOT run your full patient-level aggregation logic (that's already done in on_test_epoch_end),
# but it can help sanity-check inputs/outputs.

import matplotlib.pyplot as plt
import torch

dl = model.test_dataloader()
batch = next(iter(dl))

# batch is (image, mask, label, gidx) because you wrap WithGlobalIndex in test_dataloader
image, mask, label, gidx = batch

# forward
with torch.no_grad():
    image_dev = image.to(model.device if hasattr(model, "device") else "cpu")
    out = model(image_dev)
    if isinstance(out, (tuple, list)):
        logits = out[0]
    else:
        logits = out
    probs = torch.softmax(logits, dim=1).cpu()
    pred = probs.argmax(dim=1)

print("Batch shapes:")
print("image:", tuple(image.shape), "mask:", tuple(mask.shape), "label:", tuple(label.shape))
print("pred distribution:", torch.bincount(pred).tolist())

# Show first 4 samples (first channel only)
n_show = min(4, image.shape[0])
for i in range(n_show):
    plt.figure()
    plt.title(f"idx={i} | gidx={int(gidx[i])} | y={int(label[i])} | pred={int(pred[i])}")
    # image: (B, C, H, W) in your pickles. Show channel 0.
    img = image[i, 0].numpy()
    msk = mask[i, 0].numpy() if mask.ndim == 4 else mask[i].numpy()
    plt.imshow(img, cmap="gray")
    plt.contour(msk, levels=[0.5], linewidths=0.8)
    plt.axis("off")
    plt.show()
